# Phase 1: Financial Risk Dataset Generation

In order to fine-tune our LLM (Llama-3), we need an **Instruction-Tuning Dataset**. 

This script generates a synthetic dataset of 15,000 records containing corporate risk narratives. The goal is to train the LLM to read a narrative (the input) and output a structured JSON response classifying the **Risk Category, Risk Summary, and Severity**.

In [ ]:
import json
import random
import pandas as pd
from typing import List, Dict
import os

# Sample vocabulary for risk narratives
COMPANIES = ["Acme Corp", "Globex", "Initech", "Umbrella Corp", "Wayne Enterprises", "Stark Industries", "Massive Dynamic"]
ACTIONS = ["failed to report", "misclassified", "experienced a breach in", "identified anomalies in", "delayed the submission of", "discovered unauthorized access to"]
DOMAINS = ["Q3 financial statements", "user data privacy protocols", "anti-money laundering (AML) controls", "supply chain vendor contracts", "employee compliance training logs", "tax withholding calculations"]
CONSEQUENCES = ["leading to potential regulatory fines.", "causing a 15% drop in stock price.", "resulting in a formal SEC investigation.", "creating significant operational downtime.", "exposing sensitive customer information.", "violating internal governance policies."]

CATEGORIES = ["Financial", "Compliance", "Operational", "Strategic"]
SEVERITIES = ["Low", "Medium", "High"]


### Generate Synthetic Records
We use a basic heuristic here to label our synthetic data. During actual inference, the Fine-Tuned LLM will learn these patterns.

In [ ]:
def generate_synthetic_record() -> Dict:
    company = random.choice(COMPANIES)
    action = random.choice(ACTIONS)
    domain = random.choice(DOMAINS)
    consequence = random.choice(CONSEQUENCES)
    
    narrative = f"{company} {action} {domain}, {consequence}"
    
    # Simple heuristic for category/severity for the dummy dataset
    category = "Compliance"
    if "financial" in domain or "tax" in domain or "stock" in consequence:
        category = "Financial"
    elif "breach" in action or "downtime" in consequence:
        category = "Operational"
    
    severity = "Medium"
    if "SEC" in consequence or "fine" in consequence or "breach" in action:
        severity = "High"
    elif "delayed" in action or "training" in domain:
        severity = "Low"
        
    summary = f"Issue with {domain} causing {consequence.strip('.')}"
    
    return {
        "narrative": narrative,
        "output": json.dumps({
            "Risk Category": category,
            "Risk Summary": summary,
            "Severity": severity
        })
    }

### Execute and Save
We will generate 15,000 records and save them to `data/financial_risk_dataset.jsonl`.

In [ ]:
def generate_dataset(num_records: int = 15000, output_path: str = "data/financial_risk_dataset.jsonl"):
    os.makedirs("data", exist_ok=True)
    print(f"Generating {num_records} synthetic risk narrative records...")
    records = [generate_synthetic_record() for _ in range(num_records)]
    
    with open(output_path, 'w') as f:
        for record in records:
            f.write(json.dumps(record) + "\n")
            
    print(f"Dataset saved to {output_path}")

# Run the generation
generate_dataset()

### Preview the Data
Let's look at the first 5 records we generated.

In [ ]:
df = pd.read_json("data/financial_risk_dataset.jsonl", lines=True)
df.head()